# Overview and setup

## The app you'll build

- Every time you use a coding agent like Claude Code, Codex, or Copilot, it stores metadata about every session on your laptop.
    - The logs live in places like `~/.claude/projects/` as JSONL files, one JSON object per line.
    - They contain usage data, token counts, model names, tool calls - valuable data trapped in an awkward nested format.
- In this workshop, taught by Alena Astrakhantseva from dltHub, we turn those logs into structured tables and dashboards.
    - We do it with dlt and the dltHub AI workbench, which lets a coding agent build pipelines from natural-language prompts.
- By the end you'll have:
    1. A dlt pipeline loading local Claude Code logs into DuckDB.
    2. A marimo dashboard over that data with activity, models, tokens, and projects.
    3. A REST API pipeline pulling agent traces from a hosted API.
    4. A scheduled deployment on the dltHub Platform with a shareable dashboard.
- The architecture looks like this:

```mermaid
flowchart LR
    A[Sources] -->|dlt pipelines| B[(DuckDB)]
    B --> C[marimo dashboards]
    B --> D[dltHub Platform]
    subgraph A [Sources]
        A1[Local JSONL logs]
        A2[REST API traces]
    end
```

## Prerequisites

You'll need these accounts and tools:
- Python 3.11 or later
- [uv](https://docs.astral.sh/uv/) package manager
- A coding agent: Claude Code, Codex, or Copilot
- A dltHub Platform account (free): [app.dlthub.com](https://app.dlthub.com/)
- Some local agent logs so `~/.claude/projects/` has JSONL files to load, and if you don't have any yet, use your agent for a bit and come back.
    - To verify you have jsonl files in the claude projects directory, run the follwing command:

In [ ]:
# %%bash
# find ~/.claude/projects -type f -name "*.jsonl" | head -5

## Scaffold the workspace


- The dltHub AI workbench has its own scaffolding command, so run it in an empty folder:
- This creates a workspace with `pyproject.toml`, a `.dlt/` config directory, `.claude/` skills, and `.mcp.json` for the MCP server.
- It also creates `__deployment__.py` for cloud deployment and a virtual environment.
- When it asks to create a virtual environment and install dependencies, say yes. It runs `uv sync` for you.

In [ ]:
# %%bash
# uvx dlthub-init@latest

## Open the workspace in your agent

- Open the scaffolded folder in your coding agent.
- The agent reads the router skill and dispatches to the right toolkit when you ask it to build a pipeline.

In [ ]:
# %%bash
# uv run --active dlthub ai init

- Confirm the workbench is running

In [ ]:
# %%bash
# uv run --active dlthub ai status

In [ ]:
# %%bash
# uv run --active dlthub ai toolkit list

- DuckDB is our destination for local development.
    - It's an in-process analytical database - no server to run
    - dlt writes to a `.duckdb` file on disk. No extra setup is needed because DuckDB comes as a dependency of dlt.

# Part 1: Local logs to pipeline

- With the workspace scaffolded, we now build a dlt pipeline that reads the JSONL session transcripts from `~/.claude/projects/` and loads them into DuckDB.
- We don't write the code by hand. We tell the agent what to build, and it uses the dltHub AI workbench to write the pipeline.

## Look at the raw logs

- Open `~/.claude/projects/` and pick a `.jsonl` file. Every session is one file with one JSON object per line.
- The `type` values vary across lines. Some common ones:
    - `user`
    - `assistant`
    - `attachment`
    - `file-history-snapshot`
- The data is deeply nested, with usage objects holding token counts and message objects holding content arrays.
- This is real, valuable data - models used, tokens spent, and tools called. But it's trapped in a format that's painful to query manually.

## Build the pipeline

- Tell the agent to build a dlt pipeline for the local logs:
    > build a dlt pipeline, load data from local Claude logs as raw JSONs into DuckDB
- The agent starts with the dltHub router skill, which figures out that the data lives in files on disk.
- It installs the filesystem-pipeline toolkit on demand
    - this toolkit didn't exist in the project when you started.
    - The router pulls it in based on the data source.
- The toolkit walks the agent through the standard workflow:
    - confirm the plan
    - scaffold the pipeline
    - configure credentials
    - run it

## The pipeline the agent builds

- The pipeline uses dlt's `filesystem` source with the `read_jsonl` reader.
- The source lists files matching a glob, and the reader opens each one and yields parsed JSON records.
- dlt connects them with the pipe operator (`|`)
- The full pipeline (see [filesystem_pipeline.py](filesystem_pipeline.py)) defines a `load` function that creates the pipeline and runs it
- A few things to notice.
    - `dev_mode=True` adds a timestamp to the dataset name on every run, so each run starts fresh.
        - That's convenient during development but wasteful in production - we'll switch it off later.
    - `write_disposition="replace"` drops and reloads the table each time.

## Normalization

- When the pipeline runs, dlt doesn't just dump the raw JSON into one table.
    - It normalizes the data by inferring types, flattening nested objects, and creating child tables for nested arrays, linked by `_dlt_id` and `_dlt_parent_id`.

## View the data locally

- dlt ships with a built-in dashboard. Run it from the command line, not from the agent session.
- Make sure your pipeline ran successfully first:
    - run this at the same level of the nested uv `pyproject.toml` file for this sub-project not the root uv file
    ```bash
    uv run --active dlthub local show
    ```
- This opens a marimo dashboard that reads from the local DuckDB file.
- You can browse the schema, see how many tables exist, look at the data in each table, and run SQL queries.
- This is where you validate that the pipeline loaded what you expected.

# Part 1: Debug and build a dashboard

## Debug the pipeline

- The agent already ran the pipeline and it works, but we want to be explicit. Tell the agent:
    > debug my pipeline
- The agent finds the debug-pipeline skill in the rest-api-pipeline toolkit and installs it.
- Debugging means the agent runs the pipeline, inspects the trace, checks for errors, and fixes them until it works.
- As Alena explains in the workshop, debugging makes sure the pipeline doesn't fail and loads some data. But it can't tell you whether the data is correct.
    - That's a judgment only you can make by looking at the output.

## Schema pollution fix

- During debugging, the agent noticed that 27 tables was excessive. besides, the root table had 377 columns.
    - It identified this as schema pollution - the deeply nested JSON was exploding into too many child tables.
- The agent fixed it by setting some columns to the JSON data type instead of unnesting them into child tables. Not all data was unnested.
- Instead of 27 tables, the pipeline now creates 4 and the root table had 66 columns.
    - The deeply nested fields stay as JSON columns that you can query later with DuckDB's JSON functions.

## Build a marimo report

- Now that the pipeline is clean, tell the agent to build a dashboard:
    > build a marimo report with detailed information about my Claude Code usage
- The agent installs the data-exploration toolkit. This toolkit contains skills for profiling data, planning charts, and assembling marimo notebooks.
- First the agent profiles the data: row counts, schemas, column stats. It writes an analysis plan - a markdown file listing the questions to answer, the SQL queries, and the Altair chart code for each one. Then it assembles the notebook.

## marimo reactive notebooks

- marimo is a reactive Python notebook. Every notebook is a plain Python script, not a JSON blob. Each cell is a Python function. When you change a cell, every cell that depends on it re-runs automatically.
- Unlike Jupyter, you can't run cells in random order, so there's always a defined order, which prevents variable mix-ups.
    - This strictness makes marimo better for dashboards, because the state is always consistent.
- The [dashboard](./claude_logs_pipeline_dashboard.py) connects to the pipeline and queries the data with SQL:
- Each chart is a data cell that runs SQL and returns a DataFrame, paired with a chart cell that builds an Altair visualization.
- The dashboard shows activity over time, messages by type, models, token usage, and top projects.
- Run the dashboard:
    ```bash
    uv run --active marimo edit claude_logs_pipeline_dashboard.py
    ```

# Part 2: Ingest from a hosted API

- Part 1 loaded local logs from disk.
- But in a real organization, your agents run in the cloud and their logs live behind an API.
- Think Logfire, Langfuse, Datadog, or the Anthropic API. You can't read files from disk. You have to request the data over HTTP.
- For this workshop, Alena built a test API that serves one million fake Claude Code traces in the same structure as real logs.
    - No authentication is required, and the data is safe to share.
    - The base URL is:
        ```text
        https://test-agent-traces-api-xt2e7ottma-ew.a.run.app
        ```

## Cloud loggers

- When you build an agent, its logs live in a cloud logger.
- Services like Logfire and Langfuse collect metadata similar to the local Claude traces: usage, models, tool calls, skills used.
- To analyze this data, you request it through the logger's REST API and load it into a database.
- Each logger produces a different trace format with different keys, different nesting, different field order.
    - That's always a problem when you have several agents.
    - dlt handles the normalization for you.

## Build the pipeline

- Continue working in the same repo and tell the agent:
    > build a dlt pipeline for https://test-agent-traces-api-xt2e7ottma-ew.a.run.app/docs, for /logs endpoint, load 20k logs into DuckDB, and build a similar marimo report
- The agent installs the rest-api-pipeline toolkit. This toolkit contains skills for creating pipelines, debugging, exploring data, and applying incremental loading.
- The agent inspects the OpenAPI spec at the `/docs` URL, figures out the base URL, the pagination type, and the data selector, and writes the pipeline.
- If you've ever built a data pipeline against an API by hand, you know how much work this saves.
    - You need to know how to request the data, how to paginate, and what the output looks like. request the data, how to paginate, and what the output looks like.

## The REST API source

- The [pipeline](./rest_api_pipeline.py) describes the API as a config dictionary with `RESTAPIConfig` type.
- The `data_selector` tells dlt that the records live under the `logs` key in the response envelope, not at the top level.
- The paginator uses offset-based pagination: each request sends `limit` and `offset` query params, and dlt reads the total count from the `total` key to know when to stop.
    - A `maximum_offset` of 20000 caps the load at 20k rows without you hand-rolling pagination loops.

## Run it

- Run the pipeline to load 20k records:
    ```bash
    uv run --active python rest_api_pipeline.py
    ```
- The same normalization happens: dlt infers types, flattens nested objects like `message.content` into child tables linked by `_dlt_parent_id`.
    - The nested `usage` object becomes columns like `usage__output_tokens` with double-underscore separators.